# Lecture 14: Visualizing and Understanding

*Notebook for UMich EECS 498 (Justin Johnson), Lecture 14 "Visualizing and Understanding" (Fall 2019 slides), with CAM and Grad-CAM from Stanford CS231n 2026, Lecture 9.*

A trained CNN works, but **what is it looking for?** This lecture is a toolbox of ways to peek inside, and some fun things the same tools can do (DeepDream, style transfer).

We'll train **one small CNN** on handwritten digits and apply every technique to it.

**What you'll learn**
1. Looking at **weights** and **activations**: filters, nearest neighbors in feature space, t-SNE, maximally activating patches.
2. **Which pixels matter?** Occlusion, saliency maps, guided backprop, CAM and Grad-CAM.
3. **Optimizing the image** instead of the weights: class visualization, adversarial examples, feature inversion, DeepDream.
4. **Textures and style:** Gram matrices, texture synthesis, and neural style transfer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.datasets import load_digits

## 0. Setup: a small CNN

**Data:** the 8×8 digits, enlarged to 16×16 so the pictures are easier to see, and shifted so the average pixel is 0.

**Network:** conv 3×3 (8 filters) → ReLU → 2×2 average pool → conv 3×3 (16) → ReLU → conv 3×3 (16) → ReLU → **global average pool** → fully connected → 10 scores.

In [ ]:
digits = load_digits()
images = np.array([ndimage.zoom(img / 16.0, 2, order=1) for img in digits.images])   # 16x16
pixel_mean = images.mean()
images = images - pixel_mean
labels = digits.target
order = np.random.default_rng(0).permutation(len(images))
train_idx, test_idx = order[:1300], order[1300:]
X_train, y_train, X_test, y_test = images[train_idx], labels[train_idx], images[test_idx], labels[test_idx]
print("images:", X_train.shape)

Convolution as a sum of 9 shifted copies (the same code as Lecture 16):

In [ ]:
def conv_forward(x, W, b):
    # x: (N, C_in, H, W)   W: (C_out, C_in, 3, 3)   ->   (N, C_out, H, W), 'same' padding
    N, C_in, H, Wd = x.shape
    padded = np.pad(x, ((0, 0), (0, 0), (1, 1), (1, 1)))
    out = np.zeros((N, W.shape[0], H * Wd))
    for di in range(3):
        for dj in range(3):
            out += W[:, :, di, dj] @ padded[:, :, di:di + H, dj:dj + Wd].reshape(N, C_in, H * Wd)
    return out.reshape(N, -1, H, Wd) + b[None, :, None, None]

def conv_backward(d_out, x, W):
    N, C_in, H, Wd = x.shape
    padded = np.pad(x, ((0, 0), (0, 0), (1, 1), (1, 1)))
    d_padded, dW = np.zeros_like(padded), np.zeros_like(W)
    d_flat = d_out.reshape(N, W.shape[0], H * Wd)
    for di in range(3):
        for dj in range(3):
            shifted = padded[:, :, di:di + H, dj:dj + Wd].reshape(N, C_in, H * Wd)
            dW[:, :, di, dj] = np.sum(d_flat @ shifted.transpose(0, 2, 1), axis=0)
            d_padded[:, :, di:di + H, dj:dj + Wd] += (W[:, :, di, dj].T @ d_flat).reshape(N, C_in, H, Wd)
    return d_padded[:, :, 1:-1, 1:-1], dW, d_out.sum(axis=(0, 2, 3))

The forward pass keeps every intermediate activation (we'll want to look at them):

In [ ]:
def forward(p, x):
    # x: (N, 16, 16). Returns a dict with every layer's values.
    c = {"x": x[:, None]}
    c["z1"] = conv_forward(c["x"], p["W1"], p["b1"]);  c["a1"] = np.maximum(0, c["z1"])     # 8 x 16 x 16
    N, C, H, W = c["a1"].shape
    c["p1"] = c["a1"].reshape(N, C, H // 2, 2, W // 2, 2).mean(axis=(3, 5))                  # 8 x 8 x 8 (2x2 avg pool)
    c["z2"] = conv_forward(c["p1"], p["W2"], p["b2"]); c["a2"] = np.maximum(0, c["z2"])     # 16 x 8 x 8
    c["z3"] = conv_forward(c["a2"], p["W3"], p["b3"]); c["a3"] = np.maximum(0, c["z3"])     # 16 x 8 x 8 (last conv)
    c["pooled"] = c["a3"].mean(axis=(2, 3))                                                  # 16 (global average pool)
    c["scores"] = c["pooled"] @ p["Wf"] + p["bf"]                                            # 10 class scores
    return c

**The one function this whole notebook relies on.** The backward pass can **start from the class scores** *and/or* **inject a gradient at any layer** (`extra`). That's what lets us compute "the gradient of a neuron with respect to the image", DeepDream, feature inversion, and style transfer with the same code. The `relu_mode` option is for guided backprop (Section 6).

In [ ]:
def relu_backward(d, z, mode="normal"):
    if mode == "normal":
        return d * (z > 0)                       # ordinary backprop
    if mode == "deconvnet":
        return d * (d > 0)                       # pass only positive gradients (ignore the forward pass)
    if mode == "guided":
        return d * (z > 0) * (d > 0)             # both: guided backprop

def backward(p, c, d_scores=None, extra=None, relu_mode="normal"):
    # Returns the gradient with respect to the input image, the gradients for the weights,
    # and the gradients that arrived at a2 and a3 (used by Grad-CAM).
    extra = extra or {}
    N = c["x"].shape[0]
    g = {}
    d_a3 = np.zeros_like(c["a3"])
    if d_scores is not None:
        g["Wf"], g["bf"] = c["pooled"].T @ d_scores, d_scores.sum(axis=0)
        H, W = c["a3"].shape[2:]
        d_a3 += (d_scores @ p["Wf"].T)[:, :, None, None] / (H * W)      # back through global average pooling
    d_a3 += extra.get("a3", 0)
    d_a2, g["W3"], g["b3"] = conv_backward(relu_backward(d_a3, c["z3"], relu_mode), c["a2"], p["W3"])
    d_a2_total = d_a2 + extra.get("a2", 0)
    d_p1, g["W2"], g["b2"] = conv_backward(relu_backward(d_a2_total, c["z2"], relu_mode), c["p1"], p["W2"])
    d_a1 = np.repeat(np.repeat(d_p1, 2, axis=2), 2, axis=3) / 4 + extra.get("a1", 0)   # back through avg pooling
    d_x, g["W1"], g["b1"] = conv_backward(relu_backward(d_a1, c["z1"], relu_mode), c["x"], p["W1"])
    return d_x[:, 0], g, {"a2": d_a2_total, "a3": d_a3}

Train it (softmax loss, Adam):

In [ ]:
def softmax(s):
    e = np.exp(s - s.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

rng = np.random.default_rng(0)
shapes = {"W1": (8, 1, 3, 3), "W2": (16, 8, 3, 3), "W3": (16, 16, 3, 3), "Wf": (16, 10)}
p = {}
for name, shape in shapes.items():
    fan_in = np.prod(shape[1:]) if len(shape) == 4 else shape[0]
    p[name] = rng.normal(0, np.sqrt(2 / fan_in), size=shape)
p.update({"b1": np.zeros(8), "b2": np.zeros(16), "b3": np.zeros(16), "bf": np.zeros(10)})

m = {k: np.zeros_like(v) for k, v in p.items()}
v = {k: np.zeros_like(val) for k, val in p.items()}
for t in range(1, 1501):
    batch = rng.integers(0, len(X_train), 64)
    c = forward(p, X_train[batch])
    probs = softmax(c["scores"])
    probs[np.arange(64), y_train[batch]] -= 1
    _, grads, _ = backward(p, c, d_scores=probs / 64)
    for name in p:
        m[name] = 0.9 * m[name] + 0.1 * grads[name]
        v[name] = 0.999 * v[name] + 0.001 * grads[name] ** 2
        p[name] -= 0.01 * (m[name] / (1 - 0.9 ** t)) / (np.sqrt(v[name] / (1 - 0.999 ** t)) + 1e-8)

test_cache = forward(p, X_test)
print(f"test accuracy: {np.mean(test_cache['scores'].argmax(axis=1) == y_test):.1%}")

## 1. What's going on inside? Start with the weights

### First layer: visualize the filters
First-layer filters look directly at pixels, so we can show them as little images. AlexNet's 64 filters (64×3×11×11) and ResNet's (64×3×7×7) look like **oriented edges, blobs, and color patterns**, much like Hubel and Wiesel's cells (Lecture 1). Ours are 3×3:

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for k in range(8):
    axes[k].imshow(p["W1"][k, 0], cmap="RdBu_r")
    axes[k].axis("off")
plt.suptitle("first-layer filters (red = positive weight, blue = negative)", fontsize=9)
plt.show()

**Higher layers:** we *can* plot the filters, but they're not that interesting: they combine the previous layer's channels, not pixels, so they don't look like anything. We need other tools.

## 2. The last layer: feature space

Our network's last hidden layer (after global average pooling) is a **16-number feature vector** per image (AlexNet's FC7 is 4,096 numbers). Run the network on many images and collect the vectors.

### Nearest neighbors in feature space
In Lecture 2, nearest neighbors in **pixel** space found images that *look* alike pixel by pixel. In **feature** space, neighbors are images the network considers alike. A good test: **shifted** digits, which have very different pixels.

In [ ]:
train_features = forward(p, X_train)["pooled"]
X_test_shifted = np.array([ndimage.shift(img, (0, 2), order=0, cval=-pixel_mean) for img in X_test])   # 2 pixels right
shifted_features = forward(p, X_test_shifted)["pooled"]

def nearest_neighbor_accuracy(train_vectors, query_vectors):
    distances = ((query_vectors[:, None, :] - train_vectors[None, :, :]) ** 2).sum(axis=2)
    return np.mean(y_train[distances.argmin(axis=1)] == y_test)

print(f"1-NN on shifted test digits, pixel space:   {nearest_neighbor_accuracy(X_train.reshape(1300, -1), X_test_shifted.reshape(len(X_test), -1)):.1%}")
print(f"1-NN on shifted test digits, feature space: {nearest_neighbor_accuracy(train_features, shifted_features):.1%}")

Feature-space neighbors still find the right digit after the shift: the features describe **what** is in the image more than **where**.

### Dimensionality reduction
To *see* the feature space, squash the vectors down to 2-D. **PCA** is the simple choice; **t-SNE** is better at keeping neighbors together:

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

test_features = test_cache["pooled"]
pixels_2d = PCA(2).fit_transform(X_test.reshape(len(X_test), -1))
features_2d = TSNE(2, random_state=0, init="pca", perplexity=30).fit_transform(test_features)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, points, title in [(axes[0], pixels_2d, "PCA of raw pixels"), (axes[1], features_2d, "t-SNE of CNN features")]:
    scatter = ax.scatter(points[:, 0], points[:, 1], c=y_test, cmap="tab10", s=8)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(scatter, ax=axes[1], ticks=range(10))
plt.show()

In feature space the digits form **clean clusters**. (The lecture shows the same thing for ImageNet: thousands of photos arranged by t-SNE, with similar objects next to each other.)

## 3. Visualizing activations

A conv layer's output is a stack of **activation maps**, one per channel (AlexNet's conv5 is 128×13×13: 128 little grayscale images). Ours, for one test digit:

In [ ]:
example = 5
fig, axes = plt.subplots(2, 9, figsize=(13, 3))
axes[0, 0].imshow(X_test[example], cmap="gray_r"); axes[0, 0].set_title(f"input: {y_test[example]}", fontsize=8)
axes[1, 0].axis("off")
for k in range(16):
    ax = axes[(k + 1) // 9, (k + 1) % 9]
    ax.imshow(test_cache["a3"][example, k], cmap="gray")
    ax.set_title(f"channel {k}", fontsize=7)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("last conv layer activations (16 channels, 8x8 each)", fontsize=9)
plt.show()

Most channels are mostly dark: each one responds only to particular patterns in particular places.

### Maximally activating patches
To see **what a channel responds to**:
1. Pick a layer and a channel.
2. Run many images through the network and record that channel's values at every position.
3. Show the **image patches** around the **largest** activations.

Each position of our last conv layer sees a 12×12 patch of the 16×16 input (its receptive field), centered near pixel $(2i, 2j)$:

In [ ]:
def top_patches(channel, how_many=8, size=12):
    maps = test_cache["a3"][:, channel]                       # (N, 8, 8)
    flat = np.argsort(maps.ravel())[::-1][:how_many]          # the largest activations
    padded = np.pad(X_test, ((0, 0), (size, size), (size, size)), constant_values=-pixel_mean)
    patches = []
    for index in flat:
        n, i, j = np.unravel_index(index, maps.shape)
        r, c = 2 * i + 1 + size - size // 2, 2 * j + 1 + size - size // 2   # top-left of the receptive field
        patches.append(padded[n, r:r + size, c:c + size])
    return patches

strongest_channels = np.argsort(test_cache["a3"].max(axis=(0, 2, 3)))[::-1][:4]
fig, axes = plt.subplots(4, 8, figsize=(9, 4.8))
for row, channel in enumerate(strongest_channels):
    for col, patch in enumerate(top_patches(channel)):
        axes[row, col].imshow(patch, cmap="gray_r", vmin=-0.4, vmax=0.7)
        axes[row, col].axis("off")
    axes[row, 0].set_title(f"channel {channel}", fontsize=8, loc="left")
plt.suptitle("maximally activating patches (each row is one channel)", fontsize=9)
plt.show()

Each row shows a consistent pattern (a particular stroke, curve, or junction). In big ImageNet networks the same method finds channels for dog faces, eyes, text, and wheels.

## 4. Which pixels matter? Saliency via occlusion

**Mask part of the image** before feeding it to the CNN, and check **how much the predicted probability changes** (Zeiler & Fergus, 2014). Slide a gray square over every position and record P(correct class):

In [ ]:
def occlusion_map(image, true_class, size=4):
    H = image.shape[0]
    heat = np.zeros((H - size + 1, H - size + 1))
    occluded = []
    for i in range(H - size + 1):
        for j in range(H - size + 1):
            copy = image.copy()
            copy[i:i + size, j:j + size] = 0.0                  # 0 = the average pixel ("gray")
            occluded.append(copy)
    probs = softmax(forward(p, np.array(occluded))["scores"])[:, true_class]
    return probs.reshape(heat.shape)

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for col, example in enumerate([0, 3, 7, 12]):
    axes[0, col].imshow(X_test[example], cmap="gray_r"); axes[0, col].set_title(f"digit {y_test[example]}", fontsize=8)
    heat = occlusion_map(X_test[example], y_test[example])
    im = axes[1, col].imshow(heat, cmap="hot", vmin=0, vmax=1)
    axes[1, col].set_title(f"P(correct) {heat.min():.2f} to {heat.max():.2f}", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("dark = covering this spot hurts the prediction", fontsize=9)
plt.show()

## 5. Saliency via backprop

Faster: compute the **gradient of the (unnormalized) class score with respect to the image pixels**, and take the **absolute value** (and, for color images, the max over RGB channels). It says how much each pixel can change the score (Simonyan et al., 2014).

In [ ]:
def saliency(image, class_index):
    c = forward(p, image[None])
    d_scores = np.zeros((1, 10)); d_scores[0, class_index] = 1.0      # gradient of ONE class score
    d_image, _, _ = backward(p, c, d_scores=d_scores)
    return np.abs(d_image[0])

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for col, example in enumerate([0, 3, 7, 12]):
    axes[0, col].imshow(X_test[example], cmap="gray_r")
    axes[1, col].imshow(saliency(X_test[example], y_test[example]), cmap="gray")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("saliency maps: bright = the score is sensitive to this pixel", fontsize=9)
plt.show()

(Saliency maps can even be used for **segmentation without supervision**: run GrabCut on the saliency map to cut out the object.)

## 6. Intermediate features via (guided) backprop

**Pick a single intermediate neuron** (one value in a conv feature map) and compute **its gradient with respect to the image pixels**. Images come out nicer if you only **backprop positive gradients through each ReLU**: that's **guided backprop**. The lecture's example: a ReLU's input in the forward pass, and the gradient arriving from above.

In [ ]:
forward_input = np.array([[1, -1, 5], [2, -5, -7], [-3, 2, 4]])
upstream = np.array([[-2, 3, -1], [6, -3, 1], [2, -1, 3]])

print("backpropagation (pass where the forward input was positive):\n", relu_backward(upstream, forward_input, "normal"))
print("'deconvnet' (pass where the gradient is positive):\n", relu_backward(upstream, forward_input, "deconvnet"))
print("guided backpropagation (both):\n", relu_backward(upstream, forward_input, "guided"))

These match the slide's three tables. Now plain vs. guided backprop for the neurons behind the maximally activating patches:

In [ ]:
def neuron_gradient(image, channel, mode):
    c = forward(p, image[None])
    i, j = np.unravel_index(c["a3"][0, channel].argmax(), (8, 8))      # the neuron's strongest position
    extra = np.zeros_like(c["a3"]); extra[0, channel, i, j] = 1.0       # gradient of that ONE neuron
    d_image, _, _ = backward(p, c, extra={"a3": extra}, relu_mode=mode)
    return d_image[0]

fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for col, channel in enumerate(strongest_channels):
    best_image = X_test[test_cache["a3"][:, channel].max(axis=(1, 2)).argmax()]
    axes[0, col].imshow(best_image, cmap="gray_r"); axes[0, col].set_title(f"channel {channel}", fontsize=8)
    axes[1, col].imshow(neuron_gradient(best_image, channel, "normal"), cmap="RdBu_r")
    axes[2, col].imshow(neuron_gradient(best_image, channel, "guided"), cmap="RdBu_r")
axes[1, 0].set_ylabel("backprop"); axes[2, 0].set_ylabel("guided")
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.show()

Guided backprop keeps only the evidence that **increases** the neuron, so the picture shows the strokes the neuron likes more cleanly.

## 7. Class activation maps: CAM and Grad-CAM (from CS231n)

**CAM** (Zhou et al., 2016) works for networks that end with **global average pooling + one linear layer**, like ours. The class score is

$$S_c = \sum_k w_{k,c}\,F_k = \sum_k w_{k,c}\,\frac{1}{HW}\sum_{h,w} f_{h,w,k} = \frac{1}{HW}\sum_{h,w}\underbrace{\sum_k w_{k,c}\,f_{h,w,k}}_{M_{c,h,w}}$$

so $M_{c,h,w}$ says **how much each position contributes to class $c$**. Problem: it only works on the last conv layer.

**Grad-CAM** (Selvaraju et al., 2017) works at **any layer** $A$:
1. compute the gradient of the class score with respect to $A$;
2. average it over all positions to get one weight per channel, $\alpha_k$;
3. $M^c = \text{ReLU}\left(\sum_k\alpha_kA_k\right)$.

At the last conv layer of our network, Grad-CAM gives exactly the CAM (with negative values clipped), divided by $HW$. Check:

In [ ]:
def grad_cam(image, class_index, layer="a3"):
    c = forward(p, image[None])
    d_scores = np.zeros((1, 10)); d_scores[0, class_index] = 1.0
    _, _, layer_grads = backward(p, c, d_scores=d_scores)
    alpha = layer_grads[layer][0].mean(axis=(1, 2))                    # one weight per channel
    return np.maximum(0, np.tensordot(alpha, c[layer][0], axes=1))     # ReLU(sum_k alpha_k A_k)

def cam(image, class_index):
    a3 = forward(p, image[None])["a3"][0]
    return np.tensordot(p["Wf"][:, class_index], a3, axes=1)          # sum_k w_kc f_k

image = X_test[3]
print("Grad-CAM (last conv) == ReLU(CAM) / 64:", np.allclose(grad_cam(image, y_test[3]), np.maximum(0, cam(image, y_test[3])) / 64))

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
axes[0].imshow(image, cmap="gray_r"); axes[0].set_title(f"digit {y_test[3]}", fontsize=8)
axes[1].imshow(cam(image, y_test[3]), cmap="jet"); axes[1].set_title("CAM", fontsize=8)
axes[2].imshow(grad_cam(image, y_test[3], "a2"), cmap="jet"); axes[2].set_title("Grad-CAM at an earlier layer", fontsize=8)
wrong = (y_test[3] + 1) % 10
axes[3].imshow(cam(image, wrong), cmap="jet"); axes[3].set_title(f"CAM for class {wrong}", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

## 8. Gradient ascent: generate an image that a neuron loves

So far: find the part of a **real** image that a neuron responds to. Now: **generate a synthetic image** that maximally activates a neuron (or class score):

$$I^* = \arg\max_I\ S_c(I) - \lambda\,\|I\|_2^2$$

1. Start from an all-zero image.
2. Repeat: forward to get the scores; backprop to get the gradient of $S_c$ with respect to the pixels; take a small step **uphill**.

The $\lambda\|I\|^2$ term is a simple **"natural image" regularizer**. A better one (Yosinski et al., 2014) also **blurs the image** now and then.

In [ ]:
def visualize_class(class_index, steps=300, lr=0.5, l2=0.05, blur_every=10):
    image = np.zeros((16, 16))
    for step in range(steps):
        c = forward(p, image[None])
        d_scores = np.zeros((1, 10)); d_scores[0, class_index] = 1.0
        d_image, _, _ = backward(p, c, d_scores=d_scores)
        image += lr * (d_image[0] - 2 * l2 * image)                 # gradient ASCENT on S_c - l2 * ||I||^2
        if blur_every and step % blur_every == 0:
            image = ndimage.gaussian_filter(image, 0.5)            # the "better regularizer"
    return image

fig, axes = plt.subplots(1, 10, figsize=(13, 1.8))
for digit in range(10):
    axes[digit].imshow(visualize_class(digit), cmap="gray_r")
    axes[digit].set_title(digit); axes[digit].axis("off")
plt.suptitle("images generated to maximize each class score", fontsize=9)
plt.show()

Several of these show strokes of the digit's shape. With big ImageNet networks and stronger image priors, the same idea produces recognizable flamingos, pelicans, and beetles.

## 9. Adversarial examples

The same procedure, with a twist (Szegedy et al., 2013):
1. Start from a **real** image.
2. Pick an **arbitrary** category.
3. Modify the image (gradient ascent) to **maximize that class score**.
4. Stop as soon as the network is fooled.

In [ ]:
def make_adversarial(image, target, lr=0.02, max_steps=500):
    adversarial = image.copy()
    for step in range(max_steps):
        c = forward(p, adversarial[None])
        if c["scores"][0].argmax() == target:
            return adversarial, step
        d_scores = np.zeros((1, 10)); d_scores[0, target] = 1.0
        d_image, _, _ = backward(p, c, d_scores=d_scores)
        adversarial += lr * d_image[0] / (np.abs(d_image[0]).max() + 1e-12)    # small, normalized steps
    return adversarial, max_steps

image, true_label = X_test[0], y_test[0]
target = (true_label + 3) % 10
adversarial, steps = make_adversarial(image, target)
change = adversarial - image
print(f"true digit {true_label}; network now says {forward(p, adversarial[None])['scores'][0].argmax()} after {steps} steps")
print(f"largest pixel change: {np.abs(change).max():.2f} (pixel values range over about {image.max() - image.min():.2f})")

fig, axes = plt.subplots(1, 4, figsize=(8, 2.3))
axes[0].imshow(image, cmap="gray_r", vmin=-0.6, vmax=0.8); axes[0].set_title(f"original ({true_label})", fontsize=8)
axes[1].imshow(adversarial, cmap="gray_r", vmin=-0.6, vmax=0.8); axes[1].set_title(f"adversarial ({target})", fontsize=8)
axes[2].imshow(change, cmap="RdBu_r", vmin=-1, vmax=1); axes[2].set_title("difference", fontsize=8)        # same color scale...
axes[3].imshow(10 * change, cmap="RdBu_r", vmin=-1, vmax=1); axes[3].set_title("10x difference", fontsize=8)  # ...so 10x shows more
for ax in axes: ax.axis("off")
plt.show()

How often does this work, and how big are the changes?

In [ ]:
results = []
for example in range(30):
    image, true_label = X_test[example], y_test[example]
    adv, steps = make_adversarial(image, (true_label + 3) % 10)
    fooled = forward(p, adv[None])["scores"][0].argmax() == (true_label + 3) % 10
    results.append((fooled, np.abs(adv - image).max()))
print(f"fooled into the chosen class: {np.mean([r[0] for r in results]):.0%} of 30 images")
print(f"median largest pixel change: {np.median([r[1] for r in results]):.2f}")

For big ImageNet networks the changes are **invisible** to humans (the slides' elephant becomes a "koala" with a difference you can only see when multiplied by 10). Our tiny 16×16 network needs bigger changes, but still far less than it would take to actually draw a different digit. Adversarial examples are a big research topic (and a security problem).

## 10. Feature inversion

**Given a CNN feature vector for an image, find a new image that:**
* **matches the given features**, and
* **looks natural** (an image prior).

$$x^* = \arg\min_x\ \|\Phi(x) - \Phi_0\|^2 + \lambda\,\mathcal{R}_{TV}(x), \qquad \mathcal{R}_{TV}(x) = \sum_{i,j}(x_{i,j+1} - x_{i,j})^2 + (x_{i+1,j} - x_{i,j})^2$$

The **total variation** regularizer encourages smoothness. **Which layer we invert from matters:** low layers keep almost everything; high layers keep only what the network needs.

In [ ]:
def tv_gradient(x):
    g = np.zeros_like(x)
    dh = x[:, 1:] - x[:, :-1]; g[:, 1:] += 2 * dh; g[:, :-1] -= 2 * dh
    dv = x[1:, :] - x[:-1, :]; g[1:, :] += 2 * dv; g[:-1, :] -= 2 * dv
    return g

def invert(target_image, layer, steps=400, lr=0.05, tv=0.02, seed=0):
    target = forward(p, target_image[None])[layer]
    x = np.random.default_rng(seed).normal(0, 0.1, (16, 16))           # start from noise
    for step in range(steps):
        c = forward(p, x[None])
        extra = {layer: 2 * (c[layer] - target)}                       # gradient of ||Phi(x) - Phi_0||^2
        d_x, _, _ = backward(p, c, extra=extra)
        grad = d_x[0] + tv * tv_gradient(x)
        x -= lr * grad / (np.abs(grad).max() + 1e-12)                  # normalized gradient DEscent step
    return x

image = X_test[7]
fig, axes = plt.subplots(1, 4, figsize=(8, 2.3))
axes[0].imshow(image, cmap="gray_r"); axes[0].set_title("original", fontsize=8)
for ax, layer, name in [(axes[1], "a1", "from layer 1"), (axes[2], "a2", "from layer 2"), (axes[3], "a3", "from layer 3")]:
    reconstruction = invert(image, layer)
    ax.imshow(reconstruction, cmap="gray_r")
    ax.set_title(f"{name}\npixel error {np.mean((reconstruction - image) ** 2):.3f}", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

**What we see:** reconstructing from the first layer gives back the original almost exactly; from deeper layers, the digit's shape survives but the background and fine details get noisier (deeper layers keep what matters for recognition, not every pixel). (The slides show the same for VGG-16: relu2_2 is nearly perfect, relu5_3 keeps the elephants but not their exact look.)

## 11. DeepDream: amplify existing features

Instead of maximizing one neuron, **amplify all the activations at some layer**:
1. Forward: compute the activations at the chosen layer.
2. **Set the gradient of that layer equal to its activation.**
3. Backward: compute the gradient on the image.
4. Update the image.

This is the same as maximizing $\sum_i f_i(I)^2$: whatever the network sees a little, it's pushed to see more. The original code also **jitters** the image (random shift each step), **L1-normalizes** the gradient, and **clips** pixel values.

In [ ]:
def deep_dream(image, layer="a2", steps=60, step_size=0.02, jitter=2, seed=0):
    rng_dream = np.random.default_rng(seed)
    x = image.copy()
    for step in range(steps):
        dx, dy = rng_dream.integers(-jitter, jitter + 1, size=2)
        x = np.roll(x, (dy, dx), axis=(0, 1))                           # jitter
        c = forward(p, x[None])
        d_x, _, _ = backward(p, c, extra={layer: c[layer]})              # gradient = activation
        x += step_size * d_x[0] / (np.abs(d_x[0]).mean() + 1e-12)       # L1-normalized step
        x = np.roll(x, (-dy, -dx), axis=(0, 1))                         # undo the jitter
        x = np.clip(x, -1.0, 1.5)                                       # clip pixel values
    return x

fig, axes = plt.subplots(2, 4, figsize=(8, 4.3))
for col, example in enumerate([1, 4, 9, 15]):
    axes[0, col].imshow(X_test[example], cmap="gray_r")
    axes[1, col].imshow(deep_dream(X_test[example]), cmap="gray_r")
axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("DeepDream")
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.show()

Our network only knows strokes, so it makes the strokes bolder and adds stroke-like blobs. A network trained on ImageNet dreams dog faces, eyes, and pagodas ("the pig-snail", "the camel-bird").

## 12. Texture synthesis

**Given a small sample of a texture, can we generate a bigger image of the same texture?**

### Classic approach: nearest neighbor (Efros & Leung, 1999)
Generate pixels **one at a time, in scanline order**. For each new pixel, look at its neighborhood of **already generated** pixels, find the most similar neighborhood in the input, and **copy** that pixel.

In [ ]:
def nearest_neighbor_texture(sample, out_size=32, radius=2, seed=0):
    rng_tex = np.random.default_rng(seed)
    H, W = sample.shape
    out = np.zeros((out_size, out_size))
    out[:radius, :] = np.tile(sample[:radius], (1, out_size // W + 1))[:, :out_size]   # seed the top rows
    out[:, :radius] = np.tile(sample[:, :radius], (out_size // H + 1, 1))[:out_size]   # and left columns
    # all "L-shaped" neighborhoods (rows above + pixels to the left) in the sample
    candidates, values = [], []
    for i in range(radius, H):
        for j in range(radius, W - radius):
            candidates.append(np.concatenate([sample[i - radius:i, j - radius:j + radius + 1].ravel(), sample[i, j - radius:j]]))
            values.append(sample[i, j])
    candidates, values = np.array(candidates), np.array(values)
    for i in range(radius, out_size):
        for j in range(radius, out_size - radius):
            neighborhood = np.concatenate([out[i - radius:i, j - radius:j + radius + 1].ravel(), out[i, j - radius:j]])
            distances = ((candidates - neighborhood) ** 2).sum(axis=1)
            best = np.flatnonzero(distances <= distances.min() + 1e-9)
            out[i, j] = values[rng_tex.choice(best)]                       # copy the best match's pixel
        out[i, out_size - radius:] = out[i, out_size - 2 * radius:out_size - radius]   # fill the right edge simply
    return out

yy, xx = np.mgrid[0:16, 0:16]
stripes = (((xx + yy) % 6) < 3).astype(float)                                   # diagonal stripes
pluses = ((((yy % 5) - 2) ** 2 + ((xx % 5) - 2) ** 2) <= 1).astype(float)       # a grid of little "+" shapes

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for col, (sample, name) in enumerate([(stripes, "stripes"), (pluses, "pluses")]):
    axes[2 * col].imshow(sample, cmap="gray"); axes[2 * col].set_title(f"input: {name} (16x16)", fontsize=8)
    axes[2 * col + 1].imshow(nearest_neighbor_texture(sample, radius=3), cmap="gray"); axes[2 * col + 1].set_title("generated (32x32)", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

### Neural texture synthesis: the Gram matrix
Each CNN layer gives a $C\times H\times W$ tensor: an $H\times W$ grid of $C$-dimensional feature vectors. The **Gram matrix** averages the outer products of these vectors over all positions:

$$G_{ij} = \frac{1}{HW}\sum_k F_{ik}F_{jk} \qquad\text{(efficiently: reshape } F \text{ to } C\times HW\text{, then } G = FF^\top/HW\text{)}$$

$G_{ij}$ says "how often features $i$ and $j$ are active **together**", an (unnormalized) covariance. **It throws away where things are**, keeping only which features appear and co-occur: exactly what a texture is. Proof: shuffling the positions doesn't change it at all.

In [ ]:
def gram(activation):
    C = activation.shape[1]
    F = activation[0].reshape(C, -1)               # C x HW
    return F @ F.T / F.shape[1]                    # C x C

features = forward(p, X_test[2][None])["a2"]
shuffled = features.reshape(1, 16, 64)[:, :, np.random.default_rng(0).permutation(64)].reshape(features.shape)
print("Gram matrix shape:", gram(features).shape)
print("Gram matrix unchanged when positions are shuffled:", np.allclose(gram(features), gram(shuffled)))

**Neural texture synthesis** (Gatys et al., 2015):
1. Run the texture through a pretrained CNN; compute the **Gram matrix at each layer**.
2. Start from **random noise**.
3. Repeatedly: compute the noise image's Gram matrices, the loss = **weighted sum of squared differences** between Gram matrices, backprop to the image, and take a gradient step.

The gradient of $\|G - G_{target}\|^2$ with respect to the features is $\frac{4}{HW}(G - G_{target})F$, which we feed into `backward` at each layer:

In [ ]:
def gram_loss_gradient(activation, target_gram):
    C = activation.shape[1]
    F = activation[0].reshape(C, -1)
    G = F @ F.T / F.shape[1]
    d_F = 4 * (G - target_gram) @ F / F.shape[1]
    return np.sum((G - target_gram) ** 2), d_F.reshape(activation.shape)

def optimize_image(gradient_fn, x, steps, lr):
    # Adam on the pixels, with a step size that shrinks to 0 by the end (so the image settles down).
    m, v = np.zeros_like(x), np.zeros_like(x)
    for t in range(1, steps + 1):
        g = gradient_fn(x)
        m = 0.9 * m + 0.1 * g
        v = 0.999 * v + 0.001 * g * g
        x = x - lr * (1 - t / steps) * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
    return x

def synthesize_texture(net, texture, layers=("a1", "a2"), size=24, steps=600, lr=0.05, seed=0):
    targets = {layer: gram(forward(net, texture[None])[layer]) for layer in layers}
    def gradient(x):
        c = forward(net, x[None])
        extra = {layer: gram_loss_gradient(c[layer], targets[layer])[1] for layer in layers}
        return backward(net, c, extra=extra)[0][0]
    noise = np.random.default_rng(seed).normal(0, 0.3, (size, size))
    return optimize_image(gradient, noise, steps, lr)

texture = stripes - 0.5
digit_net_result = synthesize_texture(p, texture)
fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.3))
axes[0].imshow(texture, cmap="gray"); axes[0].set_title("texture", fontsize=8)
axes[1].imshow(digit_net_result, cmap="gray"); axes[1].set_title("synthesized with\nour digit network", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

**Not great!** There's a hint of diagonal structure, but mostly noise. The reason: our digit network has only **8** first-layer channels, so its first Gram matrix is just 8×8 = 64 numbers, far too few statistics to pin down a texture. VGG-19 has 64 to 512 channels per layer.

**A surprising fact:** for texture synthesis, the filters don't even need to be trained. A **wide network with random filters** works well (Ustyuzhaninov et al., 2017). Our `forward` and `backward` work for any number of channels, so let's make one with 64 channels per layer:

In [ ]:
rng_wide = np.random.default_rng(5)
W_ = 64
wide_net = {"W1": rng_wide.normal(0, 1, (W_, 1, 3, 3)), "b1": rng_wide.normal(0, 0.1, W_),
            "W2": rng_wide.normal(0, np.sqrt(2 / (9 * W_)), (W_, W_, 3, 3)), "b2": np.zeros(W_),
            "W3": rng_wide.normal(0, np.sqrt(2 / (9 * W_)), (W_, W_, 3, 3)), "b3": np.zeros(W_),
            "Wf": np.zeros((W_, 10)), "bf": np.zeros(10)}       # random filters; the classifier part isn't used

fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
for row, sample in enumerate([stripes - 0.5, pluses - 0.5]):
    axes[row, 0].imshow(sample, cmap="gray"); axes[row, 0].set_title("texture (16x16)", fontsize=8)
    for col, layers in zip([1, 2, 3], [("a1",), ("a1", "a2"), ("a1", "a2", "a3")]):
        axes[row, col].imshow(synthesize_texture(wide_net, sample, layers), cmap="gray")
        axes[row, col].set_title(f"Gram of {', '.join(layers)} (24x24)", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.show()

Now the synthesized images (bigger than the sample!) have the right **texture**: diagonal stripes, and scattered "+" shapes, but **not** an exact copy of the layout. With VGG-19, Gram matrices from higher layers recover larger and larger structures of the texture (radishes, pebbles, palm trees). Applied to a painting, the "texture" is the artwork's **style**.

## 13. Neural style transfer

**Feature reconstruction + Gram reconstruction** (Gatys et al., 2016): find an image that matches
* the **features** of a **content image** (what's where), and
* the **Gram matrices** of a **style image** (the texture),

minimizing a weighted sum of the two losses. More weight on content keeps the original look; more weight on style makes it more like the texture.

In [ ]:
def style_transfer(content_image, style_image, style_weight, content_layer="a1", style_layers=("a1", "a2"), steps=500, lr=0.03, tv=0.05):
    content_target = forward(wide_net, content_image[None])[content_layer]
    style_targets = {layer: gram(forward(wide_net, style_image[None])[layer]) for layer in style_layers}
    def gradient(x):
        c = forward(wide_net, x[None])
        extra = {layer: style_weight * gram_loss_gradient(c[layer], style_targets[layer])[1] for layer in style_layers}
        content_grad = 2 * (c[content_layer] - content_target) * 1000 / content_target.size     # feature loss
        extra[content_layer] = extra.get(content_layer, 0) + content_grad
        return backward(wide_net, c, extra=extra)[0][0] + tv * tv_gradient(x)                  # + smoothness
    return optimize_image(gradient, content_image.copy(), steps, lr)      # start from the content image

content = X_test[7]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
axes[0].imshow(content, cmap="gray_r"); axes[0].set_title("content", fontsize=8)
axes[1].imshow(stripes - 0.5, cmap="gray_r"); axes[1].set_title("style", fontsize=8)
for ax, weight in zip(axes[2:], [0.01, 0.1, 1, 10]):
    result = style_transfer(content, stripes - 0.5, weight)
    predicted = forward(p, result[None])["scores"][0].argmax()          # what does our DIGIT network see?
    ax.imshow(result, cmap="gray_r"); ax.set_title(f"style weight {weight}\ndigit network says: {predicted}", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

**What we see:** with a small style weight the output is just the content; around weight 1 we get **the digit drawn in diagonal stripes**, and our digit network still recognizes it; with a large style weight the stripes take over and the digit is lost. (The lecture shows the same trade-off with a portrait: more content weight on one side, more style weight on the other.)

### Fast style transfer
**Problem:** style transfer needs **many** forward/backward passes through VGG for every image: very slow. **Solution** (Johnson, Alahi & Fei-Fei, 2016): **train another neural network** to do it in one forward pass, using the same content and style losses (computed by the pretrained CNN) as its training loss.

Two normalization tricks came out of this line of work:
* **Instance normalization** (Lecture 7) was developed **for style transfer**: replacing batch norm with instance norm makes the results better.
* **Conditional instance normalization** (Dumoulin et al., 2017): **one network for many styles**. Learn a separate scale $\gamma_s$ and shift $\beta_s$ **per style**; everything else is shared. Mixing the $\gamma, \beta$ of two styles even blends them.

In [ ]:
def conditional_instance_norm(x, gammas, betas, style):
    # x: (C, H, W). Normalize each channel over space, then apply THIS style's scale and shift.
    mean = x.mean(axis=(1, 2), keepdims=True)
    std = x.std(axis=(1, 2), keepdims=True) + 1e-5
    return gammas[style][:, None, None] * (x - mean) / std + betas[style][:, None, None]

features = forward(p, X_test[7][None])["a2"][0]                          # 16 x 8 x 8
gammas = {"van gogh": rng.uniform(0.5, 2, 16), "picasso": rng.uniform(0.5, 2, 16)}
betas = {"van gogh": rng.normal(0, 1, 16), "picasso": rng.normal(0, 1, 16)}
for style in gammas:
    out = conditional_instance_norm(features, gammas, betas, style)
    print(f"{style:9s}: channel 0 mean {out[0].mean():+.2f}, std {out[0].std():.2f}  "
          f"(= beta {betas[style][0]:+.2f}, gamma {gammas[style][0]:.2f})")

Same network, same features, different per-style numbers: that's all it takes to switch styles.

## 14. In PyTorch (optional, run in Colab)
In PyTorch, "the gradient with respect to the image" is just `requires_grad=True` on the input:

In [ ]:
# [PyTorch] Needs: pip install torch torchvision (downloads pretrained weights)
import torch, torchvision
model = torchvision.models.resnet18(weights="DEFAULT").eval()
image = torch.rand(1, 3, 224, 224, requires_grad=True)          # use a real (normalized) photo here
score = model(image)[0].max()                                   # the top class score
score.backward()
saliency_map = image.grad.abs().max(dim=1).values               # absolute value, max over RGB
print("saliency map:", tuple(saliency_map.shape))

## Summary
**Many methods for understanding CNN representations:**
* **Activations:** nearest neighbors in feature space, dimensionality reduction (PCA, t-SNE), maximally activating patches, occlusion.
* **Gradients:** saliency maps, (guided) backprop, CAM and Grad-CAM, class visualization, adversarial ("fooling") images, feature inversion.
* **Fun:** DeepDream, texture synthesis, and style transfer.

The key trick behind the gradient methods: **backprop all the way to the pixels, and optimize the image instead of the weights.**

### Check your understanding
1. Why are nearest neighbors in feature space more meaningful than in pixel space?
2. What's the difference between a saliency map and occlusion?
3. What does guided backprop change compared with normal backprop?
4. Why does the Gram matrix capture texture but not layout?
5. Style transfer starts from the content image and minimizes two losses. What does each loss preserve?

<details><summary>Answers</summary>

1. Features describe what the network recognizes (robust to shifts and small changes); pixel distances are thrown off by shifts and lighting.
2. A saliency map uses one backward pass (gradient of the score w.r.t. the pixels); occlusion actually masks regions and reruns the network, which is slower but measures real changes in the output.
3. At each ReLU it passes only gradients that are positive *and* where the forward input was positive.
4. It averages feature co-occurrences over all positions, so shuffling positions doesn't change it (shown in Section 12).
5. The feature (content) loss preserves what is where; the Gram (style) loss preserves the texture statistics of the style image.
</details>

**Next:** Lecture 15, object detection.